# Problem 3 - Phan tich va phan khuc khach hang
Notebook dung du lieu Silver that, chi tinh don delivered; moi bang va bieu do hien thi truc tiep khi chay.

In [ ]:
from pathlib import Path
import warnings
try:
    from IPython.display import display
except ImportError:
    display = print
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def locate_repo():
    starts = [Path.cwd().resolve(), Path(r'C:/dpbngoc/DA & AI/DAAI_N2.3')]
    for start in starts:
        for path in [start, *start.parents]:
            if (path / 'silver_data').exists():
                return path
    raise FileNotFoundError('Khong tim thay thu muc silver_data')

REPO_ROOT = locate_repo()
def find_data_file(filename):
    candidates = sorted((REPO_ROOT / 'silver_data').glob(f'.silver_pipeline_work_*/excel/{filename}'), reverse=True)
    candidates += [REPO_ROOT / filename, REPO_ROOT / 'silver_data' / filename]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(filename)

def show_table(title, df):
    print(f'\n{"="*100}\n{title}\n{"="*100}')
    display(df)

def show_result(df, name, index=False):
    show_table(name.replace('_',' ').replace('.csv','').upper(), df)

def main():
    orders = pd.read_csv(find_data_file('orders_enriched.csv'), low_memory=False)
    items = pd.read_csv(find_data_file('order_items.csv'), low_memory=False)
    products = pd.read_csv(find_data_file('products.csv'), low_memory=False)
    customers = pd.read_csv(find_data_file('customers.csv'), low_memory=False)
    orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
    orders['order_status'] = orders['order_status'].astype(str).str.lower().str.strip()
    orders = orders[orders['order_status'].eq('delivered')].copy()
    items[['quantity','unit_price','discount_amount']] = items[['quantity','unit_price','discount_amount']].apply(pd.to_numeric, errors='coerce')
    items['NetRevenue'] = (items['quantity'] * items['unit_price'] - items['discount_amount'].fillna(0)).clip(lower=0)
    fact = items.merge(orders[['order_id','order_date','customer_id']], on='order_id', how='inner', validate='many_to_one')
    fact = fact.merge(products[['product_id','product_name','category','segment']], on='product_id', how='left', validate='many_to_one')
    order_level = fact.groupby(['customer_id','order_id','order_date'], as_index=False).agg(OrderRevenue=('NetRevenue','sum'), BasketSize=('quantity','sum'), DistinctProducts=('product_id','nunique'))
    order_level = order_level.sort_values(['customer_id','order_date','order_id'])
    order_level['DaysBetweenPurchase'] = order_level.groupby('customer_id')['order_date'].diff().dt.days

    customer = order_level.groupby('customer_id', as_index=False).agg(
        Frequency=('order_id','nunique'), Monetary=('OrderRevenue','sum'),
        AOV=('OrderRevenue','mean'), BasketSize=('BasketSize','mean'),
        AvgDaysBetweenPurchase=('DaysBetweenPurchase','mean'),
        FirstPurchase=('order_date','min'), LastPurchase=('order_date','max'))
    observation_end = order_level['order_date'].max()
    customer['RecencyDays'] = (observation_end - customer['LastPurchase']).dt.days
    customer['TenureDays'] = (customer['LastPurchase'] - customer['FirstPurchase']).dt.days.clip(lower=1)
    f_median, m_median = customer['Frequency'].median(), customer['Monetary'].median()
    customer['CustomerSegment'] = np.select([
        (customer['Frequency'] >= f_median) & (customer['Monetary'] >= m_median),
        (customer['Frequency'] >= f_median) & (customer['Monetary'] < m_median),
        (customer['Frequency'] < f_median) & (customer['Monetary'] >= m_median)],
        ['Tan suat cao - Gia tri cao','Tan suat cao - Gia tri thap','Tan suat thap - Gia tri cao'],
        default='Tan suat thap - Gia tri thap')
    customer['CustomerType'] = np.where(customer['Frequency'] >= 2, 'Repeat Customer', 'One-time Buyer')
    fallback_interval = (customer['TenureDays'] / customer['Frequency'].sub(1).replace(0, np.nan)).clip(lower=1)
    customer['PurchaseIntervalDays'] = customer['AvgDaysBetweenPurchase'].fillna(fallback_interval).fillna(365).clip(lower=1)
    customer['HistoricalCLV'] = customer['Monetary']
    customer['Predicted12M_CLV'] = customer['AOV'] * (365 / customer['PurchaseIntervalDays']).clip(upper=24)
    customer['CLVRank'] = customer['Predicted12M_CLV'].rank(method='dense', ascending=False).astype(int)
    customer['CLVSegment'] = pd.qcut(customer['Predicted12M_CLV'].rank(method='first'), q=4, labels=['Low','Potential','High','VIP'])
    clv_actions = {
        'VIP':'Chuong trinh VIP, uu tien dich vu, early access va cross-sell cao cap',
        'High':'Loyalty tier, combo theo category ua thich va referral reward',
        'Potential':'Voucher lan mua tiep, goi y bo sung va nhac mua dung chu ky',
        'Low':'Welcome/reactivation offer chi phi thap; kiem soat tan suat khuyen mai'}
    customer['CLVRecommendation'] = customer['CLVSegment'].astype(str).map(clv_actions)
    customer = customer.merge(customers, on='customer_id', how='left', validate='one_to_one')
    show_result(customer, 'customer_segmentation_clv')

    repeat_summary = customer.groupby('CustomerType', as_index=False).agg(Customers=('customer_id','nunique'), Revenue=('Monetary','sum'), AOV=('AOV','mean'), BasketSize=('BasketSize','mean'))
    repeat_summary['CustomerShare_%'] = repeat_summary['Customers'] / repeat_summary['Customers'].sum() * 100
    repeat_summary['RevenueShare_%'] = repeat_summary['Revenue'] / repeat_summary['Revenue'].sum() * 100
    show_result(repeat_summary, 'repeat_vs_one_time')

    fact['month'] = fact['order_date'].dt.to_period('M').dt.to_timestamp()
    fact['day_of_week'] = fact['order_date'].dt.day_name()
    monthly = fact.groupby('month', as_index=False).agg(Revenue=('NetRevenue','sum'), Orders=('order_id','nunique'), Customers=('customer_id','nunique'), Quantity=('quantity','sum'))
    monthly['AOV'] = monthly['Revenue'] / monthly['Orders']
    category = fact.groupby('category', as_index=False).agg(Revenue=('NetRevenue','sum'), Orders=('order_id','nunique'), Quantity=('quantity','sum'), Customers=('customer_id','nunique')).sort_values('Revenue', ascending=False)
    weekday = fact.groupby('day_of_week', as_index=False).agg(Revenue=('NetRevenue','sum'), Orders=('order_id','nunique'), Quantity=('quantity','sum'))
    customer_type = customer[['customer_id','CustomerType','CustomerSegment','CLVSegment']]
    behavior = fact.merge(customer_type, on='customer_id', how='left', validate='many_to_one')
    repeat_category = behavior.groupby(['CustomerType','category'], as_index=False).agg(Revenue=('NetRevenue','sum'), Orders=('order_id','nunique'), Quantity=('quantity','sum'))
    segment_category = behavior.groupby(['CustomerSegment','category'], as_index=False).agg(Revenue=('NetRevenue','sum'), Orders=('order_id','nunique'))
    preferred = segment_category.loc[segment_category.groupby('CustomerSegment')['Revenue'].idxmax()].rename(columns={'category':'PreferredCategory'})
    action_plan = preferred[['CustomerSegment','PreferredCategory','Revenue','Orders']].copy()
    action_plan['PromotionStrategy'] = action_plan['CustomerSegment'].map({
        'Tan suat cao - Gia tri cao':'Exclusive bundle va loyalty reward, khong giam gia dai tra',
        'Tan suat cao - Gia tri thap':'Combo tang AOV va nguong freeship',
        'Tan suat thap - Gia tri cao':'Reminder theo chu ky va uu dai lan mua tiep',
        'Tan suat thap - Gia tri thap':'Welcome/reactivation voucher co gioi han'})
    action_plan['ProductStrategy'] = 'Uu tien goi y san pham trong ' + action_plan['PreferredCategory'].astype(str) + '; cross-sell category lien quan'
    show_result(monthly, 'behavior_monthly'); show_result(category, 'behavior_category')
    show_result(weekday, 'behavior_weekday'); show_result(repeat_category, 'repeat_category')
    show_result(segment_category, 'segment_category'); show_result(action_plan, 'segment_action_plan')

    peak = pd.DataFrame([
        {'Metric':'Peak revenue month','Value':str(monthly.loc[monthly['Revenue'].idxmax(),'month'].date())},
        {'Metric':'Peak order month','Value':str(monthly.loc[monthly['Orders'].idxmax(),'month'].date())},
        {'Metric':'Peak weekday','Value':str(weekday.loc[weekday['Orders'].idxmax(),'day_of_week'])},
        {'Metric':'Top category by revenue','Value':str(category.iloc[0]['category'])}])
    show_result(peak, 'peak_behavior_summary')

    plt.figure(figsize=(12,5)); plt.plot(monthly['month'], monthly['Revenue'])
    plt.title('Doanh thu khach hang theo thang'); plt.ylabel('VND'); plt.tight_layout()
    plt.show()
    plt.figure(figsize=(8,5)); plt.scatter(customer['Frequency'], customer['Monetary'], s=8, alpha=.35)
    plt.axvline(f_median, color='gray', linestyle='--'); plt.axhline(m_median, color='gray', linestyle='--')
    plt.xlabel('Frequency'); plt.ylabel('Monetary'); plt.title('Phan khuc khach hang theo 2 chi so')
    plt.tight_layout(); plt.show()
    repeat_summary.set_index('CustomerType')[['CustomerShare_%','RevenueShare_%']].plot(kind='bar', figsize=(8,5), title='Repeat vs One-time')
    plt.ylabel('Ty trong (%)'); plt.tight_layout(); plt.show()
    repeat_summary.set_index('CustomerType')[['AOV','BasketSize']].plot(kind='bar', subplots=True, figsize=(8,7), title=['AOV theo loai khach hang','Basket Size theo loai khach hang'])
    plt.tight_layout(); plt.show()
    clv_summary = customer.groupby('CLVSegment', observed=True, as_index=False).agg(Customers=('customer_id','nunique'), HistoricalCLV=('HistoricalCLV','sum'), Predicted12M_CLV=('Predicted12M_CLV','sum'))
    show_table('TONG HOP 4 NHOM CLV', clv_summary)
    clv_summary.set_index('CLVSegment')[['HistoricalCLV','Predicted12M_CLV']].plot(kind='bar', figsize=(9,5), title='CLV theo phan khuc')
    plt.ylabel('Gia tri'); plt.tight_layout(); plt.show()
    category.head(15).sort_values('Revenue').plot(kind='barh', x='category', y='Revenue', figsize=(9,6), legend=False, title='Top Category theo doanh thu')
    plt.tight_layout(); plt.show()
    weekday.plot(kind='bar', x='day_of_week', y='Orders', figsize=(9,5), legend=False, title='So don theo ngay trong tuan')
    plt.tight_layout(); plt.show()
    repeat_chart = repeat_category.pivot(index='category', columns='CustomerType', values='Revenue').fillna(0)
    repeat_chart.plot(kind='bar', figsize=(10,5), title='Doanh thu Repeat va One-time theo Category')
    plt.tight_layout(); plt.show()
    print('Hoan tat Problem 3')
    print('So khach hang:', len(customer), '| Repeat rate:', round(float((customer['Frequency'] >= 2).mean() * 100), 2), '%')
    print('Tat ca bang va bieu do da hien thi truc tiep trong notebook/terminal.')
    return customer, repeat_summary, monthly, category, action_plan

customer_result, repeat_result, monthly_result, category_result, action_plan_result = main()
